# Day 22 — Pandas: **Document Metadata Processing**

**Phase 1 · Module 3: Prompt Engineering & RAG** · ~30 min

Every chunk you embed carries **metadata** — source file, page, product, date, access level. RAG quality lives and dies on this: filter by `access_level` and you enforce data governance; filter by `product` and you cut irrelevant retrieval in half. Pandas is how you clean and reshape that metadata before it reaches the vector store.

Four steps:
1. `pd.read_csv` a 100-chunk metadata table.
2. Clean — strip whitespace, drop duplicates, fill missing.
3. `pd.json_normalize` to flatten nested metadata.
4. Export to **Parquet** for efficient columnar storage.

> Needs `pandas` (+ `pyarrow` for Parquet). No API keys. `pip install pandas pyarrow` if a cell reports the module missing.

## 1. Load a metadata table

Ingestion pipelines emit a manifest — one row per chunk. We build a messy 100-row CSV in memory (real whitespace, dupes, and gaps included) and read it with `pd.read_csv`, the function you'll point at S3 or a manifest file in production.

In [1]:
import pandas as pd
from io import StringIO
import random

random.seed(7)
products = ["Mortgage", "Personal Loan", "Overdraft", "Credit Card", "Savings"]
rows = ["doc_id,product,page,access_level,source"]
for i in range(100):
    p = random.choice(products)
    page = random.choice([1, 2, 3, "", 5])          # some blank pages
    lvl = random.choice([" internal ", "public", "Internal", ""])  # messy casing/space
    rows.append(f"chunk_{i:03d},{p},{page},{lvl},policy_{p.replace(' ','_').lower()}.pdf")
rows.append(rows[1])   # exact copy of first data row -> guaranteed duplicate for the dedupe demo

df = pd.read_csv(StringIO("\n".join(rows)))
print("shape:", df.shape)
print(df.head(4).to_string(index=False))
print("\ndtypes:\n", df.dtypes)

shape: (101, 5)
   doc_id   product  page access_level               source
chunk_000 Overdraft   2.0          NaN policy_overdraft.pdf
chunk_001  Mortgage   1.0    internal   policy_mortgage.pdf
chunk_002 Overdraft   5.0    internal  policy_overdraft.pdf
chunk_003   Savings   2.0    internal    policy_savings.pdf

dtypes:
 doc_id              str
product             str
page            float64
access_level        str
source              str
dtype: object


☝ 101 rows read (100 + the appended duplicate). Note `page` came in as `object`, not `int` — the blank values forced pandas to keep the column as strings. That's the first thing cleaning fixes.

## 2. Clean — strip, dedupe, fill

Three defects in almost every real manifest:
- **Whitespace / casing** — `" internal "` vs `"Internal"` split what should be one group. `.str.strip().str.lower()`.
- **Duplicates** — re-ingested chunks. `drop_duplicates()`.
- **Missing values** — blank `page`, blank `access_level`. `fillna` with a safe default; **never leave `access_level` null** — default it to the *most restrictive* value.

In [2]:
# strip + normalise the string columns
for col in ["product", "access_level", "source"]:
    df[col] = df[col].astype(str).str.strip().str.lower()

# blank -> NaN, then fill
df["access_level"] = df["access_level"].replace("", pd.NA).fillna("internal")   # safe default = restrictive
df["page"] = pd.to_numeric(df["page"], errors="coerce").fillna(0).astype(int)   # blank pages -> 0, column now int

before = len(df)
df = df.drop_duplicates()
print(f"dropped {before - len(df)} duplicate row(s) -> {len(df)} rows")
print("\naccess_level counts:\n", df["access_level"].value_counts())
print("\npage dtype now:", df["page"].dtype)

dropped 1 duplicate row(s) -> 100 rows

access_level counts:
 access_level
internal    80
public      20
Name: count, dtype: int64

page dtype now: int64


☝ `access_level` collapsed to a clean set (`internal`/`public`) with no nulls — every chunk is now governable. `page` is a real `int`. Deduplication removed the re-ingested row. Defaulting missing access to `internal` (restrictive) is the governance-safe choice: a metadata gap must never *widen* who can see a chunk.

## 3. Flatten nested metadata with `json_normalize`

Chunkers often emit metadata as a **nested JSON blob** per chunk (`{"embedding": {...}, "doc": {...}}`). A vector store wants flat columns. `pd.json_normalize` explodes nested keys into dotted columns in one call.

In [3]:
nested = [
    {"doc_id": "chunk_000",
     "meta": {"embedding": {"model": "all-MiniLM-L6-v2", "dim": 384},
              "doc": {"product": "mortgage", "reviewed": True}}},
    {"doc_id": "chunk_001",
     "meta": {"embedding": {"model": "titan-v2", "dim": 1024},
              "doc": {"product": "overdraft", "reviewed": False}}},
]

flat = pd.json_normalize(nested, sep=".")
print(flat.to_string(index=False))
print("\ncolumns:", list(flat.columns))

   doc_id meta.embedding.model  meta.embedding.dim meta.doc.product  meta.doc.reviewed
chunk_000     all-MiniLM-L6-v2                 384         mortgage               True
chunk_001             titan-v2                1024        overdraft              False

columns: ['doc_id', 'meta.embedding.model', 'meta.embedding.dim', 'meta.doc.product', 'meta.doc.reviewed']


☝ `meta.embedding.dim` and `meta.doc.product` became flat columns — no manual dict-digging. `sep="."` sets the join character; each nesting level becomes one dotted column, ready to merge onto the chunk table or write straight to the store.

## 4. Export to Parquet

CSV re-parses every value as text on load and stores no types. **Parquet** is columnar and typed: it keeps `page` as int, compresses per-column, and lets the reader load *only* the columns it needs — the format Bedrock Knowledge Bases and every data lake use for chunk metadata.

In [4]:
import os, tempfile
path = os.path.join(tempfile.gettempdir(), "chunks_meta.parquet")
try:
    df.to_parquet(path, index=False)                 # needs pyarrow
    back = pd.read_parquet(path, columns=["doc_id", "product", "access_level"])
    csv_bytes = len(df.to_csv(index=False).encode())
    pq_bytes = os.path.getsize(path)
    print(f"parquet written: {path}")
    print(f"csv  size: {csv_bytes:5d} bytes")
    print(f"pqt  size: {pq_bytes:5d} bytes  (typed + compressed)")
    print(f"\ncolumn-pruned read (3 of {df.shape[1]} cols):")
    print(back.head(3).to_string(index=False))
except Exception as e:
    print("pyarrow not installed -> pip install pyarrow")
    print("repr:", repr(e))

parquet written: /var/folders/jd/b91jfh0n5r90fqg_0_2t9xg00000gn/T/chunks_meta.parquet
csv  size:  5308 bytes
pqt  size:  4252 bytes  (typed + compressed)

column-pruned read (3 of 5 cols):
   doc_id   product access_level
chunk_000 overdraft     internal
chunk_001  mortgage     internal
chunk_002 overdraft     internal


☝ Parquet preserves dtypes (no re-inferring `page` on every read) and lets `read_parquet(columns=[...])` pull just the 3 columns a governance filter needs — you never load the embedding blob to check an access level. That column-pruning is why manifests ship as Parquet, not CSV.

## 5. Recap — metadata is a governance surface

| Step | Call | Why |
|---|---|---|
| **Load** | `pd.read_csv` | one row per chunk; watch dtypes on blanks |
| **Normalise** | `.str.strip().str.lower()` | merge `" Internal "` and `"internal"` |
| **Fill safely** | `fillna("internal")` on access | a null must never widen visibility |
| **Dedupe** | `drop_duplicates()` | re-ingested chunks pollute retrieval counts |
| **Flatten** | `pd.json_normalize(sep=".")` | nested blob → flat columns |
| **Store** | `to_parquet` | typed, compressed, column-prunable |
